```
목표 및 포부
캐글에서 새로운 데이터셋 가져온다(xml 파일 포함된 데이터셋)
그걸 가져와서 내가 요리조리 수정해서 txt로 변환하여
YOLOv8 모델이 학습하기 좋게 만든다.
의존도는 최소한으로 가져가면서..

0. Pandas로 CSV 불러오기: pd.read_csv() 활용
1. 클래스명 -> ID 매핑 Dict 만들기: ({'apple': 0, 'banana': 1})
2. 이미지 파일별로 grouping하기: df.groupby('image_id')를 사용하면 이미지 1개에 속한 모든 바운딩 박스를 한번에 처리
3. PIL 또는 OpenCV로 이미지 너비 높이 구하기
4. 정규화 공식 적용 후 .txt 파일로 저장하기

```

In [14]:
import pandas as pd
import os

In [32]:

class_map = {0: 'face'}

csv_path = "dataset/human_faces/human-faces-object-detection/faces.csv"
df = pd.read_csv(csv_path)
# df.head()
# 	image_name  	width	height	x0	y0	x1	 y1
# 0	00001722.jpg	1333	2000	490	320	687	 664
# 1	00001044.jpg	2000	1333	791	119	1200 436
# 2	00001050.jpg	667	    1000	304	155	407	 331
# 3	00001736.jpg	626	    417	    147	14	519	 303
# 4	00003121.jpg	626	    418	    462	60	599	 166

# print(df.loc[0, 'width'])
# print(df.loc[0]['width'])

# df.groupby('image_name')

# print(df.head())


#        image_name  width  height   x0  y0   x1  y1
# 586  00000003.jpg    500     350  101  25  176  87

for img_name, group in df.groupby('image_name'):
    # group 안에는 특정 img_name에 해당하는 행들이 들어있음.

    # if len(group) >= 2:
    #     print(f"얼굴이 여러 개 있는 이미지: {img_name} ({len(group)}개)")

    # 텍스트 파일에 쓸 한줄 한줄을 모을 리스트
    txt_lines = []

    # group 내부의 각 행을 순회
    for idx, row in group.iterrows():
        w_img = row['width']
        h_img = row['height']
        xmin, ymin, xmax, ymax = row['x0'], row['y0'], row['x1'], row['y1']

        x_center = (xmin + xmax) / (w_img * 2)
        y_center = (ymin + ymax) / (h_img * 2)
        w = (xmax - xmin) / w_img
        h = (ymax - ymin) / h_img

        line = f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"
        txt_lines.append(line)

    pure_name, ext = os.path.splitext(img_name)
    txt_path = f"dataset/human_faces/human-faces-object-detection/labels/{pure_name}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write('\n'.join(txt_lines) + '\n')



In [15]:
path = "dataset/human_faces/human-faces-object-detection/images"
print(len(os.listdir(path)))

2205
